# 01 — Sanity check

End-to-end smoke check that the refactored `lad` package produces correct outputs on one class.
Run this notebook on a machine that has the data tree available (laptop with the original dataset symlinked, or HPC after Step 4 has populated NPZs).

Each cell is intentionally short and prints a clear pass/fail line.

In [ ]:
# 1. Imports + config
import os, sys
from pathlib import Path
import numpy as np
import torch

# add repo root to path so `import paths` and `import lad` work in-place
REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))
import paths
from lad import decomposition, metrics, backbones, data

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BACKBONE = 'resnet34'
NPZ_VARIANT = 'ViT-B-16_r16_14x14'
CLASS = os.environ.get('LAD_SMOKE_CLASS', 'african_elephant')
FOLD = 0
RANK = 25
EPOCHS = 25  # smoke version (paper trains for 150)
print(f'device={DEVICE}  class={CLASS}  fold={FOLD}  rank={RANK}')

In [ ]:
# 2. Backbone loads, feature shape matches expected
torch.manual_seed(42); np.random.seed(42)
model = backbones.load_backbone(BACKBONE, device=DEVICE)
g, h_2d = backbones.make_g_and_h2d(model, BACKBONE, device=DEVICE)
with torch.inference_mode():
    z = g(torch.zeros(1, 3, 224, 224, device=DEVICE))
C, H, W = backbones.feature_dims(BACKBONE)
assert tuple(z.shape) == (1, C, H, W), f'expected (1,{C},{H},{W}), got {tuple(z.shape)}'
print(f'g(zeros).shape == {tuple(z.shape)}  ✓')

In [ ]:
# 3. Build paired (image, NPZ) items for one class
images_root = paths.FILTERED_ROOT / BACKBONE / 'correct'
npz_root = paths.NPZ_ROOT / BACKBONE / f'clip_local_P_{NPZ_VARIANT}' / 'correct'
suffix = data.npz_suffix_from_variant(NPZ_VARIANT)
classes, _c2i, paired, report = data.build_paired_items(images_root, npz_root, suffix, classes=[CLASS])
print('report:', report)
assert paired, f'no paired items found for class {CLASS!r}'
print(f'len(paired) = {len(paired)}  ✓')

In [ ]:
# 4. NPZ batch shapes are correct, S is non-negative
from torch.utils.data import DataLoader, Subset
ds = data.ImagenetImageNpzDataset(paired)
loader = DataLoader(Subset(ds, list(range(min(8, len(ds))))), batch_size=8, collate_fn=data.collate_concept_npz)
imgs, ys, P, scores, masks, _concepts, _meta, ids = next(iter(loader))
assert imgs.shape[0] == ys.shape[0] == P.shape[0]
assert P.shape[1] == RANK, f'NPZ rank {P.shape[1]} ≠ expected {RANK}'
assert torch.all(P.float() >= 0), 'CLIP S must be non-negative'
assert not torch.isnan(P.float()).any()
print(f'P shape = {tuple(P.shape)}, scores = {tuple(scores.shape)}, masks = {tuple(masks.shape)}  ✓')

In [ ]:
# 5. LAD trainer monotone-on-average + non-negative W
import torch.nn.functional as F
device = DEVICE
with torch.inference_mode():
    Z = g(imgs.to(device))                                  # [B, C, h, w]
    A_flat, _idx, hw = decomposition.flatten_hw_rowmajor(Z)
    P_dev = P.float().to(device)
    if P_dev.shape[-2:] != hw:
        P_dev = F.adaptive_avg_pool2d(P_dev, hw)
    B = imgs.size(0)
    S_flat = P_dev.permute(0, 2, 3, 1).reshape(B * hw[0] * hw[1], RANK)

W, losses = decomposition.train_W_pgd(A_flat.float(), S_flat.float(), rank=RANK, n_iter=EPOCHS)
assert torch.all(W >= 0), 'W must be non-negative'
assert losses[-1] <= losses[0] * 1.05, f'loss did not decrease: {losses[0]:.3e} -> {losses[-1]:.3e}'
print(f'loss[0]={losses[0]:.4e}  loss[-1]={losses[-1]:.4e}  W min/max=({W.min().item():.4f}, {W.max().item():.4f})  ✓')

In [ ]:
# 6. Inference path: solve_S_hat is non-negative and reduces error vs init
S_hat0 = decomposition.nonneg_ls_init_U(A_flat.float(), W)
S_hat = decomposition.solve_S_hat(A_flat.float(), W, pgd_iters=30)
assert torch.all(S_hat >= 0)
err0 = ((S_hat0 @ W.T - A_flat.float()) ** 2).mean().item()
err1 = ((S_hat  @ W.T - A_flat.float()) ** 2).mean().item()
print(f'closed-form MSE = {err0:.4e}   +PGD MSE = {err1:.4e}   delta = {err0 - err1:+.4e}  ✓')

In [ ]:
# 7. Metric ranges look sane (Gini in [0,1], C-Ins AUC ≥ 0)
U_per_image = S_hat.reshape(B, hw[0] * hw[1], RANK).mean(dim=1)
imp = metrics.estimate_importance(U_per_image, W.T.cpu().numpy(), h_2d, h_2d, int(ys[0].item()),
                                  batch_size=64, number_of_concepts=RANK, device=device)
g_idx = metrics.calculate_gini(imp)
spar = metrics.compute_sparsity(U_per_image)
assert 0.0 <= g_idx <= 1.0,  f'gini out of range: {g_idx}'
assert 0.0 <= spar <= 1.0
print(f'gini={g_idx:.3f}  sparsity={spar:.3f}  importance min/max=({imp.min():.3e}, {imp.max():.3e})  ✓')

In [ ]:
# 8. (Optional) regression test against a pre-refactor W_final.pt
ref_path = REPO.parent / 'ECCV' / 'models' / 'checkpoints' / 'Final_run_with_KL_epoch_200' / \
           f'{BACKBONE}_clip_local_P_{NPZ_VARIANT}_correct' / f'fold_{FOLD}' / CLASS / 'W_final.pt'
if ref_path.exists():
    W_ref = torch.load(ref_path, map_location='cpu').float()
    if W_ref.shape == W.shape:
        rel = (W_ref - W.cpu()).norm().item() / W_ref.norm().item()
        print(f'relative Frobenius diff vs reference: {rel:.4e}  (tolerance 1e-1 since we trained 25 epochs vs 200)')
    else:
        print(f'shape mismatch: ref={tuple(W_ref.shape)}, new={tuple(W.shape)} — likely different rank or transpose')
else:
    print(f'no reference W found at {ref_path} — skipping regression test')